# Allscripts Touchworks — Note Hydration

Populates `_exponent.omop_tw.note` from Allscripts Touchworks clinical result/report notes.

## Source Tables
- `_bronze_allscripts_tw_works.dbo_order_activity_header` — note metadata (patient, encounter, visit, date)
- `_bronze_allscripts_tw_works.dbo_order_result_mapper` — links order activity header to result activity header
- `_bronze_allscripts_tw_works.dbo_result_text` — note body text

## Join Chain
`dbo_order_activity_header.ID` → `dbo_order_result_mapper.OrderActivityHeaderID`
→ `dbo_order_result_mapper.ResultActivityHeaderID` = `dbo_result_text.ResultID`

## Pipeline
1. `silver_note` — staged temp view, full OMOP field set
2. MERGE → `omop_silver.note`
3. INSERT → `omop_mapping.source_to_note`
4. `gold` — resolves surrogate IDs and FK references
5. MERGE → `omop_tw.note`

## Dependencies
- `omop_mapping.source_to_person` must be populated for allscripts_tw
- `omop_mapping.source_to_visit_occurrence` must be populated for allscripts_tw

In [0]:
%sql
TRUNCATE TABLE _exponent.omop_tw.note;

In [0]:
%sql
DELETE FROM _exponent.omop_silver.note
WHERE source_system = 'allscripts_tw';

In [0]:
%sql
DELETE FROM _exponent.omop_mapping.source_to_note
WHERE source_system = 'allscripts_tw';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS _exponent.omop_silver.note (
  note_source_value             STRING    NOT NULL,
  person_id                     BIGINT    NOT NULL,
  note_date                     DATE      NOT NULL,
  note_datetime                 TIMESTAMP,
  note_type_concept_id          INT       NOT NULL,
  note_class_concept_id         INT       NOT NULL,
  note_title                    STRING,
  note_text                     STRING    NOT NULL,
  encoding_concept_id           INT       NOT NULL,
  language_concept_id           INT       NOT NULL,
  provider_id                   BIGINT,
  visit_occurrence_source_value STRING,
  visit_detail_id               BIGINT,
  note_event_id                 BIGINT,
  note_event_field_concept_id   BIGINT,
  source_system                 STRING    NOT NULL,
  last_mod_tsp                  TIMESTAMP
);

CREATE TABLE IF NOT EXISTS _exponent.omop_mapping.source_to_note (
  note_id           BIGINT    GENERATED ALWAYS AS IDENTITY,
  source_system     STRING    NOT NULL,
  note_source_value STRING    NOT NULL,
  active_flag       BOOLEAN   NOT NULL,
  created_tsp       TIMESTAMP NOT NULL,
  last_mod_tsp      TIMESTAMP NOT NULL,
  merge_id          BIGINT,
  merge_reason      STRING
);

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW silver_note AS
SELECT
  CONCAT_WS(
    CHR(31),
    'allscripts_tw',
    'dbo_order_activity_header',
    'id',
    CAST(oah.ID AS BIGINT)
  )                                                     AS note_source_value,  -- staging key

  stp.person_id                                         AS person_id,

  CAST(oah.CreateDTTM AS DATE)                          AS note_date,
  oah.CreateDTTM                                        AS note_datetime,

  44814645                                              AS note_type_concept_id,   -- EHR Note
  44814645                                              AS note_class_concept_id,  -- Clinical Note (default; no LOINC classification available from source)

  NULL                                                  AS note_title,
  rt.ResultText                                         AS note_text,

  32678                                                 AS encoding_concept_id,    -- UTF-8
  4180186                                               AS language_concept_id,    -- English

  NULL                                                  AS provider_id,

  -- Visit staging key for resolution in gold
  CASE
    WHEN oah.VisitID IS NOT NULL AND oah.VisitID <> 0
    THEN CONCAT_WS(
           CHR(31),
           'allscripts_tw',
           'dbo_visit',
           'id',
           CAST(oah.VisitID AS BIGINT)
         )
    ELSE NULL
  END                                                   AS visit_occurrence_source_value,

  NULL                                                  AS visit_detail_id,

  NULL                                                  AS note_event_id,
  NULL                                                  AS note_event_field_concept_id,

  'allscripts_tw'                                       AS source_system

FROM _exponent._bronze_allscripts_tw_works.dbo_order_activity_header oah

JOIN _exponent._bronze_allscripts_tw_works.dbo_order_result_mapper orm
  ON orm.OrderActivityHeaderID = oah.ID

JOIN _exponent._bronze_allscripts_tw_works.dbo_result_text rt
  ON rt.ResultID = orm.ResultActivityHeaderID

LEFT JOIN _exponent.omop_mapping.source_to_person stp
  ON stp.person_source_value = CONCAT_WS(
       CHR(31),
       'allscripts_tw',
       'dbo_person',
       'id',
       CAST(oah.PatientID AS BIGINT)
     )
 AND stp.active_flag = TRUE

WHERE rt.ResultText IS NOT NULL
  AND oah.PatientID  IS NOT NULL
  AND stp.person_id  IS NOT NULL;

In [0]:
%sql
MERGE INTO _exponent.omop_silver.note AS target
USING (
  SELECT *
  FROM (
    SELECT
      *,
      ROW_NUMBER() OVER (
        PARTITION BY note_source_value
        ORDER BY note_date DESC
      ) AS rn
    FROM silver_note
  )
  WHERE rn = 1
) AS source
ON target.note_source_value = source.note_source_value

WHEN MATCHED AND NOT (
     target.person_id                  <=> source.person_id
 AND target.note_date                  <=> source.note_date
 AND target.note_datetime              <=> source.note_datetime
 AND target.note_type_concept_id       <=> source.note_type_concept_id
 AND target.note_class_concept_id      <=> source.note_class_concept_id
 AND target.note_title                 <=> source.note_title
 AND target.note_text                  <=> source.note_text
 AND target.encoding_concept_id        <=> source.encoding_concept_id
 AND target.language_concept_id        <=> source.language_concept_id
 AND target.provider_id                <=> source.provider_id
 AND target.visit_occurrence_source_value <=> source.visit_occurrence_source_value
 AND target.visit_detail_id            <=> source.visit_detail_id
 AND target.note_event_id             <=> source.note_event_id
 AND target.note_event_field_concept_id <=> source.note_event_field_concept_id
 AND target.source_system              <=> source.source_system
) THEN UPDATE SET
  target.person_id                     = source.person_id,
  target.note_date                     = source.note_date,
  target.note_datetime                 = source.note_datetime,
  target.note_type_concept_id          = source.note_type_concept_id,
  target.note_class_concept_id         = source.note_class_concept_id,
  target.note_title                    = source.note_title,
  target.note_text                     = source.note_text,
  target.encoding_concept_id           = source.encoding_concept_id,
  target.language_concept_id           = source.language_concept_id,
  target.provider_id                   = source.provider_id,
  target.visit_occurrence_source_value = source.visit_occurrence_source_value,
  target.visit_detail_id               = source.visit_detail_id,
  target.note_event_id                 = source.note_event_id,
  target.note_event_field_concept_id   = source.note_event_field_concept_id,
  target.source_system                 = source.source_system,
  target.last_mod_tsp                  = CURRENT_TIMESTAMP()

WHEN NOT MATCHED THEN INSERT (
  note_source_value,
  person_id,
  note_date,
  note_datetime,
  note_type_concept_id,
  note_class_concept_id,
  note_title,
  note_text,
  encoding_concept_id,
  language_concept_id,
  provider_id,
  visit_occurrence_source_value,
  visit_detail_id,
  note_event_id,
  note_event_field_concept_id,
  source_system,
  last_mod_tsp
) VALUES (
  source.note_source_value,
  source.person_id,
  source.note_date,
  source.note_datetime,
  source.note_type_concept_id,
  source.note_class_concept_id,
  source.note_title,
  source.note_text,
  source.encoding_concept_id,
  source.language_concept_id,
  source.provider_id,
  source.visit_occurrence_source_value,
  source.visit_detail_id,
  source.note_event_id,
  source.note_event_field_concept_id,
  source.source_system,
  CURRENT_TIMESTAMP()
);

In [0]:
%sql
INSERT INTO _exponent.omop_mapping.source_to_note (
    source_system,
    note_source_value,
    active_flag,
    created_tsp,
    last_mod_tsp,
    merge_id,
    merge_reason
)
SELECT
    s.source_system,
    s.note_source_value,
    TRUE                AS active_flag,
    CURRENT_TIMESTAMP() AS created_tsp,
    CURRENT_TIMESTAMP() AS last_mod_tsp,
    NULL                AS merge_id,
    NULL                AS merge_reason
FROM (
    SELECT DISTINCT
        source_system,
        note_source_value
    FROM _exponent.omop_silver.note
    WHERE note_source_value IS NOT NULL
      AND source_system = 'allscripts_tw'
) s
LEFT ANTI JOIN _exponent.omop_mapping.source_to_note x
  ON s.note_source_value = x.note_source_value
 AND x.source_system = 'allscripts_tw';

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW gold AS
SELECT
  stn.note_id,
  n.person_id,
  n.note_date,
  n.note_datetime,
  n.note_type_concept_id,
  n.note_class_concept_id,
  n.note_title,
  n.note_text,
  n.encoding_concept_id,
  n.language_concept_id,
  n.provider_id,
  stvo.visit_occurrence_id,
  n.visit_detail_id,
  n.note_event_id,
  n.note_event_field_concept_id,
  n.note_source_value

FROM _exponent.omop_silver.note n

JOIN _exponent.omop_mapping.source_to_note stn
  ON n.note_source_value = stn.note_source_value
 AND stn.source_system   = 'allscripts_tw'
 AND stn.active_flag     = TRUE

LEFT JOIN _exponent.omop_mapping.source_to_visit_occurrence stvo
  ON n.visit_occurrence_source_value = stvo.visit_occurrence_source_value
 AND stvo.source_system = 'allscripts_tw'
 AND stvo.active_flag   = TRUE

WHERE n.source_system = 'allscripts_tw';

In [0]:
%sql
MERGE INTO _exponent.omop_tw.note AS target
USING gold AS source
ON target.note_id = source.note_id

WHEN MATCHED AND NOT (
     target.person_id             <=> source.person_id
 AND target.note_date             <=> source.note_date
 AND target.note_datetime         <=> source.note_datetime
 AND target.note_type_concept_id  <=> source.note_type_concept_id
 AND target.note_class_concept_id <=> source.note_class_concept_id
 AND target.note_title            <=> source.note_title
 AND target.note_text             <=> source.note_text
 AND target.encoding_concept_id   <=> source.encoding_concept_id
 AND target.language_concept_id   <=> source.language_concept_id
 AND target.provider_id           <=> source.provider_id
 AND target.visit_occurrence_id   <=> source.visit_occurrence_id
 AND target.visit_detail_id             <=> source.visit_detail_id
 AND target.note_event_id               <=> source.note_event_id
 AND target.note_event_field_concept_id <=> source.note_event_field_concept_id
 AND target.note_source_value     <=> source.note_source_value
) THEN UPDATE SET
  target.person_id             = source.person_id,
  target.note_date             = source.note_date,
  target.note_datetime         = source.note_datetime,
  target.note_type_concept_id  = source.note_type_concept_id,
  target.note_class_concept_id = source.note_class_concept_id,
  target.note_title            = source.note_title,
  target.note_text             = source.note_text,
  target.encoding_concept_id   = source.encoding_concept_id,
  target.language_concept_id   = source.language_concept_id,
  target.provider_id           = source.provider_id,
  target.visit_occurrence_id   = source.visit_occurrence_id,
  target.visit_detail_id             = source.visit_detail_id,
  target.note_event_id               = source.note_event_id,
  target.note_event_field_concept_id = source.note_event_field_concept_id,
  target.note_source_value     = source.note_source_value

WHEN NOT MATCHED THEN INSERT (
  note_id,
  person_id,
  note_date,
  note_datetime,
  note_type_concept_id,
  note_class_concept_id,
  note_title,
  note_text,
  encoding_concept_id,
  language_concept_id,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  note_event_id,
  note_event_field_concept_id,
  note_source_value
) VALUES (
  source.note_id,
  source.person_id,
  source.note_date,
  source.note_datetime,
  source.note_type_concept_id,
  source.note_class_concept_id,
  source.note_title,
  source.note_text,
  source.encoding_concept_id,
  source.language_concept_id,
  source.provider_id,
  source.visit_occurrence_id,
  source.visit_detail_id,
  source.note_event_id,
  source.note_event_field_concept_id,
  source.note_source_value
);